# Notebook 2: Camera Adversarial Attacks

Demonstrate all 8 camera attack types on IR and EO cameras.

**Attacks:** FGSM, PGD, BIM, C&W, Universal, Backdoor, Physical, EOT

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from data_loader import SensorDataLoader
from attacks.camera_attacks import CameraAdversarialAttacker, AttackType
from visualization.plot_utils import plot_attack_impact

%matplotlib inline

## 2.1 Load Data

In [ ]:
SCENARIO = 'scenario2'
DATA_PATH = f'../data/sensor_fusion_dataset/{SCENARIO}'

loader = SensorDataLoader(DATA_PATH)
detections = loader.load_all_detections()

# Focus on IR Camera (sensor_id=3) and EO Camera (sensor_id=4)
ir_detections = detections[3]
eo_detections = detections[4]

print(f"IR Camera: {len(ir_detections)} detections")
print(f"EO Camera: {len(eo_detections)} detections")

## 2.2 Initialize Attacker

In [ ]:
attacker = CameraAdversarialAttacker(epsilon=0.05, max_iterations=50)
print("Available attacks:", [a.name for a in AttackType])

## 2.3 Run Individual Attacks on IR Camera

In [ ]:
attacks_to_demo = [
    AttackType.FGSM,
    AttackType.PGD,
    AttackType.BIM,
    AttackType.CW,
    AttackType.UNIVERSAL,
    AttackType.BACKDOOR,
    AttackType.PHYSICAL,
    AttackType.EOT
]

results = {}
for attack_type in attacks_to_demo:
    attacked = attacker.attack_detections(ir_detections.copy(), attack_type, sensor_id=3)
    results[attack_type.name] = attacked
    print(f"{attack_type.name}: {len(attacked)} detections")

## 2.4 Visualize Attack Impact (Bearings)

In [ ]:
# Compare benign vs attacked bearings
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, (attack_name, attacked_df) in enumerate(results.items()):
    ax = axes[idx]
    
    # Extract bearings
    benign_bearings = ir_detections['bearing'].values
    attacked_bearings = attacked_df['bearing'].values
    
    ax.hist(benign_bearings, bins=50, alpha=0.5, label='Benign', density=True)
    ax.hist(attacked_bearings, bins=50, alpha=0.5, label='Attacked', density=True)
    
    # Compute shift
    shift = np.mean(np.abs(attacked_bearings - benign_bearings))
    
    ax.set_title(f'{attack_name}\nMean shift: {shift:.4f} rad')
    ax.set_xlabel('Bearing (rad)')
    ax.set_ylabel('Density')
    ax.legend()
    ax.grid(True)

plt.suptitle('Camera Attack Impact on IR Camera Bearings', fontsize=14)
plt.tight_layout()
plt.show()

## 2.5 Spatial Impact Visualization

In [ ]:
# Show FGSM attack on 2D positions
attack_name = 'FGSM'
attacked_df = results[attack_name]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Benign
ax1.scatter(ir_detections['x_piren'], ir_detections['y_piren'], s=5, alpha=0.5, c='blue')
ax1.set_title('Benign IR Camera Detections')
ax1.set_xlabel('East (m)')
ax1.set_ylabel('North (m)')
ax1.grid(True)
ax1.set_aspect('equal')

# Attacked
ax2.scatter(attacked_df['x_piren'], attacked_df['y_piren'], s=5, alpha=0.5, c='red')
ax2.set_title(f'{attack_name} Attacked IR Camera Detections')
ax2.set_xlabel('East (m)')
ax2.set_ylabel('North (m)')
ax2.grid(True)
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

## 2.6 Epsilon Sweep Analysis

In [ ]:
epsilons = np.linspace(0.01, 0.2, 10)
mean_shifts = []

for eps in epsilons:
    att = CameraAdversarialAttacker(epsilon=eps)
    attacked = att.attack_detections(ir_detections.copy(), AttackType.FGSM, sensor_id=3)
    shift = np.mean(np.abs(attacked['bearing'].values - ir_detections['bearing'].values))
    mean_shifts.append(shift)

plt.figure(figsize=(10, 6))
plt.plot(epsilons, mean_shifts, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Epsilon (perturbation magnitude)')
plt.ylabel('Mean bearing shift (rad)')
plt.title('FGSM Attack: Epsilon vs Bearing Shift')
plt.grid(True)
plt.show()

## 2.7 Compare IR vs EO Camera Attack Sensitivity

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for col, attack_type in enumerate(attacks_to_demo[:4]):
    # IR Camera
    attacked_ir = attacker.attack_detections(ir_detections.copy(), attack_type, sensor_id=3)
    shift_ir = np.mean(np.abs(attacked_ir['bearing'].values - ir_detections['bearing'].values))
    
    ax = axes[0, col]
    ax.hist(ir_detections['bearing'], bins=50, alpha=0.5, label='Benign', density=True)
    ax.hist(attacked_ir['bearing'], bins=50, alpha=0.5, label='Attacked', density=True)
    ax.set_title(f'IR - {attack_type.name}\nShift: {shift_ir:.4f}')
    ax.legend()
    ax.grid(True)
    
    # EO Camera
    attacked_eo = attacker.attack_detections(eo_detections.copy(), attack_type, sensor_id=4)
    shift_eo = np.mean(np.abs(attacked_eo['bearing'].values - eo_detections['bearing'].values))
    
    ax = axes[1, col]
    ax.hist(eo_detections['bearing'], bins=50, alpha=0.5, label='Benign', density=True)
    ax.hist(attacked_eo['bearing'], bins=50, alpha=0.5, label='Attacked', density=True)
    ax.set_title(f'EO - {attack_type.name}\nShift: {shift_eo:.4f}')
    ax.legend()
    ax.grid(True)

plt.suptitle('Camera Attack Comparison: IR vs EO', fontsize=14)
plt.tight_layout()
plt.show()